# 코드 설명

- **코드 파일 사용 목차** : 데이터를 훈련 데이터와 테스트 데이터로 분리하기
- **해당 코드에서 사용하는 CSV 파일** : `north_korea_missile_test_database.csv`
    - 규모 : 135행 × 19열
    - 목표 변수 : `Missile Name` (미사일 명칭, 문자열 범주)
    - 설명 변수 : 나머지 18열 (발사 일시, 발사 시설, 위경도, 사거리 등)
    - 결측 : 19열 중 7열에 결측값 존재
- **사용 라이브러리**
    - `pandas` (as pd) — CSV 적재 및 표 형식 데이터 처리
    - `sklearn` — 데이터 분할
- **사용되는 함수**
    - `pd.read_csv("파일경로")`
        - 함수의 목적 : CSV 파일을 `DataFrame` 객체로 적재
        - 함수의 파라미터
            - `filepath_or_buffer` : 파일 경로. 첫 번째 위치 인자
            - `sep` : 열 구분자. 기본값 `,`
            - `encoding` : 문자 인코딩. 한글 포함 시 `utf-8` 또는 `cp949` 지정
    - `DataFrame.drop(labels, axis)`
        - 함수의 목적 : 지정한 행 또는 열을 제거한 새 `DataFrame` 반환
        - 함수의 파라미터
            - `labels` : 제거 대상 이름. 문자열 또는 리스트
            - `axis` : `0`이면 행, `1`이면 열. 기본값 `0`
            - `inplace` : `True`면 원본 직접 수정. 기본값 `False`
    - `train_test_split(*arrays, test_size, random_state, stratify)`
        - 함수의 목적 : 배열을 무작위로 섞은 뒤 두 부분으로 분할
        - 함수의 파라미터
            - `*arrays` : 분할 대상. `X`, `y`를 함께 전달하면 행 대응 관계 유지
            - `test_size` : 뒤쪽 부분의 비율. `0.2`면 20%
            - `random_state` : 난수 시드. 고정 시 실행마다 동일 분할
            - `stratify` : 지정한 배열의 범주 비율을 보존하며 분할
        - 반환값 : `X_train, X_test, y_train, y_test` 순서의 객체 4개
    - `len(객체)`
        - 함수의 목적 : 행 개수 확인. `DataFrame`은 행 수, `Series`는 원소 수 반환
- **실행 순서** : 적재 → X·y 분리 → 1차 분할(훈련 80% / 테스트 20%) → 2차 분할(훈련에서 검증 분리) → 결과 확인

In [1]:
# ── Google Colab 준비 셀 ─────────────────────────────────────────
# 로컬(VS Code, JupyterLab)에서 실행하면 아무 동작 없이 넘어갑니다.
# Colab에서 실행하면 저장소를 내려받아 실습 폴더로 이동합니다.
import os
import subprocess
import sys

if "google.colab" in sys.modules:
    저장소 = "https://github.com/hedgehog-TechnicalSeminar/python-cyber-seminar.git"
    이름 = 저장소.rsplit("/", 1)[-1].removesuffix(".git")
    실습폴더 = "01-data-preprocessing"

    if not os.path.isdir(f"/content/{이름}"):
        subprocess.run(["git", "clone", "-q", 저장소], cwd="/content", check=True)
    os.chdir(f"/content/{이름}/{실습폴더}")

    # Colab에는 pandas, numpy, scikit-learn이 이미 설치되어 있습니다.
    # 추가 패키지가 필요한 회차에서는 아래 줄의 주석을 해제하세요.
    # subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "../requirements.txt"], check=True)

    print("Colab 준비 완료 :", os.getcwd())
else:
    print("로컬 환경 — 준비 셀을 건너뜁니다.")

로컬 환경 — 준비 셀을 건너뜁니다.


In [2]:
from sklearn.model_selection import train_test_split
import pandas as pd

df = pd.read_csv("north_korea_missile_test_database.csv")
y = df["Missile Name"]
X = df.drop("Missile Name", axis=1)


### 1. 데이터 적재 및 X, y 분리

**목적**

지도학습은 입력값으로부터 정답을 예측하는 규칙을 데이터에서 찾는 방식. 학습에 앞서 원본 표를 **문제에 해당하는 X**와 **정답에 해당하는 y**로 분리 필요.

| 기호 | 명칭 | 형태 | 내용 |
|---|---|---|---|
| `X` | 특징 행렬 | 135행 × 18열 | 예측에 사용할 설명 변수 |
| `y` | 목표 벡터 | 135개 | 예측 대상인 `Missile Name` |

**각 줄의 동작**

- `pd.read_csv()` — CSV 텍스트를 `DataFrame`으로 적재
- `df["Missile Name"]` — 열 1개 추출. 반환 자료형은 `DataFrame`이 아닌 1차원 `Series`
- `df.drop("Missile Name", axis=1)` — 목표 변수 열을 제외한 나머지 전체를 X로 지정. `axis=1`이 열 방향이고 `axis=0`은 행 방향이므로 혼동 주의

**대소문자 표기**

X는 행과 열을 가진 2차원 구조이므로 대문자, y는 값이 한 줄로 나열된 1차원 구조이므로 소문자 사용. scikit-learn 문서와 예제 전반이 동일 규약 사용.

**본 데이터의 특성**

- 표본 135개, 목표 변수가 문자열 범주이므로 **다중 분류** 과제에 해당
- 설명 변수 18열 중 대부분이 문자열이며 7개 열에 결측값 존재
- 따라서 실제 모델 학습 전에 결측 처리와 범주형 인코딩 필요. 본 파일은 분할 단계까지만 취급

In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=31
)

### 2. 1차 분할 — 테스트 데이터 격리

**분할이 필요한 이유**

학습에 사용한 데이터로 성능을 측정하면 실제보다 높은 값 산출. 모델이 이미 그 데이터에 맞춰 조정된 상태이기 때문. 성능을 제대로 측정하려면 학습에 전혀 쓰지 않은 데이터가 별도로 필요하며, 이를 미리 떼어 두는 방식이 **홀드아웃(hold-out)**.

**분할 결과**

135개 → 훈련 108개(80%) + 테스트 27개(20%)

**파라미터**

| 파라미터 | 기능 |
|---|---|
| `test_size=0.2` | 뒤쪽(테스트) 비율을 20%로 지정 |
| `random_state=31` | 난수 시드 고정 |

`random_state` 미지정 시 실행할 때마다 분할 결과 변동. 이 경우 성능 변화가 모델 개선 때문인지 분할이 바뀐 탓인지 구분 불가. 따라서 재현성 확보를 위해 시드 고정 필요. 값 자체에 의미는 없으며 실험 전반에 동일 값 사용이 요건.

**반환값 순서**

`X_train, X_test, y_train, y_test` 순서. X 두 개가 y 두 개보다 앞. 순서를 잘못 받아도 오류가 발생하지 않고 잘못된 학습으로 이어지므로 주의.

**보완 사항**

`stratify` 미지정으로 단순 무작위 추출 수행. 목표 변수의 범주 분포가 치우친 경우 특정 범주가 한쪽에만 몰릴 수 있으므로, 분류 과제에서는 `stratify=y` 지정으로 범주 비율 보존 권장.

In [4]:
X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train, test_size=0.25, random_state=31
)

### 3. 2차 분할 — 검증 데이터 분리

**전체가 아닌 잔여분을 분할**

본 셀의 분할 대상은 전체 135개가 아니라 1차 분할 후 남은 108개. 따라서 `test_size`도 전체 기준이 아닌 **108개 기준 비율**로 해석.

전체의 20%를 검증용으로 확보하려면 다음과 같이 계산.

$$\text{test\_size} = \frac{0.2}{1 - 0.2} = 0.25, \qquad 108 \times 0.25 = 27 = 135 \times 0.2$$

`0.2`를 그대로 넣으면 108 × 0.2 ≈ 21개가 되어 의도한 비율과 불일치.

**최종 배분**

| 데이터 | 크기 | 전체 대비 | 용도 |
|---|---|---|---|
| `X_train` | 81 | 60% | 모델 학습 |
| `X_val` | 27 | 20% | 설정 비교 및 선택 |
| `X_test` | 27 | 20% | 최종 성능 측정 |

**셋으로 나누는 이유**

검증 데이터는 하이퍼파라미터를 바꿔 가며 어느 설정이 나은지 비교하는 용도이므로 반복 참조. 그 과정에서 검증 데이터에 잘 맞는 설정이 선택되며, 결과적으로 검증 성능은 실제 성능보다 높게 산출. 이를 검증 데이터에 대한 과적합 또는 **선택 편향**이라 지칭.

따라서 선택 과정에 전혀 관여하지 않은 데이터를 따로 두고 마지막에 **한 번만** 사용. 테스트 결과를 보고 모델을 다시 손보는 행위는 그 시점에서 테스트 데이터를 검증 데이터로 격하시키며, **데이터 누수**에 해당.

**대안**

표본이 135개로 적어 검증용 27개만으로는 측정값의 변동 폭이 큼. 이 경우 데이터를 k개 조각으로 나눠 번갈아 검증하는 **k-겹 교차검증**이 데이터 활용 면에서 유리.

**주의 — 셀 재실행**

반환값을 다시 `X_train`, `y_train`에 할당하므로 기존 108개는 소실. 이 셀만 다시 실행하면 81개가 재차 분할되어 61개로 축소되므로, 노트북은 첫 셀부터 순차 실행 권장.

In [5]:
print(f"원래 데이터셋의 크기   : {len(X):3}")
print(f"훈련 데이터셋의 크기 : {len(X_train):3} = "
    f"{len(X_train)}/{len(X)} = {len(X_train)/len(X)*100:5.2f}%")
print(f"검증 데이터셋의 크기 : {len(X_val):3} = "
    f"{len(X_val)}/{len(X)} = {len(X_val)/len(X)*100:5.2f}%")
print(f"테스트 데이터셋의 크기 : {len(X_test):3} = "
    f"{len(X_test)}/{len(X)} = {len(X_test)/len(X)*100:5.2f}%")

원래 데이터셋의 크기   : 135
훈련 데이터셋의 크기 :  81 = 81/135 = 60.00%
검증 데이터셋의 크기 :  27 = 27/135 = 20.00%
테스트 데이터셋의 크기 :  27 = 27/135 = 20.00%


### 4. 분할 결과 확인

**목적**

분할 비율이 의도대로인지 확인하는 점검 단계. 분할 오류는 오류 메시지 없이 넘어간 뒤 성능 이상으로 뒤늦게 드러나므로, 다음 단계로 넘어가기 전에 수치로 확인하는 편이 안전.

**출력**

```
원래 데이터셋의 크기   : 135
훈련 데이터셋의 크기 :  81 = 81/135 = 60.00%
검증 데이터셋의 크기 :  27 = 27/135 = 20.00%
테스트 데이터셋의 크기 :  27 = 27/135 = 20.00%
```

81 + 27 + 27 = 135로 총합 일치, 비율도 설계값 6:2:2와 일치.

**f-문자열 서식**

f-문자열의 `{}` 안은 `{표현식:서식}` 구조이며, 콜론 뒤가 출력 형태 지정.

| 서식 | 의미 | 출력 |
|---|---|---|
| `{len(X)}` | 서식 없이 값만 삽입 | `135` |
| `{len(X_train):3}` | 최소 3칸 확보. 숫자는 우측 정렬 | `· 81` |
| `{...*100:5.2f}` | 최소 5칸, 소수점 아래 2자리 | `60.00` |

칸 수를 맞추면 자릿수가 다른 숫자의 끝이 세로로 정렬되어 비교 용이.

**자주 발생하는 오류**

`{len(X_train)/{len(X)}}`처럼 중괄호를 겹쳐 쓰면 안쪽 `{len(X)}`가 값이 아닌 **집합(set)**으로 해석. 그 결과 정수와 집합의 나눗셈이 시도되어 다음 오류 발생.

```
TypeError: unsupported operand type(s) for /: 'int' and 'set'
```

`81/135` 형태로 출력하려면 두 값에 각각 중괄호를 씌워 `{len(X_train)}/{len(X)}`로 작성. 이때 슬래시는 나눗셈 기호가 아닌 출력되는 문자.